# Fig. S2 — extrusion times and cluster/random ratio

Left and middle columns: `plot_extrusion_time_heatmap` (random / cluster). Right column: `plot_extrusion_time_ratio_heatmap`. Top row: $t_{50\%}$; bottom row: $t_{5\%}$.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
from vertex_lite.custom import load_data_v3 as load_data
import pyprojroot.here as here

sns.set_theme(
    style="white",
    context="paper",
    rc={
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "axes.facecolor": "#faf9f7",
        "figure.facecolor": "#faf9f7",
    }
)

output_dir = Path(here()) / "results" / "supplementary_figures"
os.makedirs(output_dir, exist_ok=True)

## Extrusion-time heatmaps (random and cluster)

In [ ]:
def plot_extrusion_time_heatmap(
    arrangement="random",
    remaining_fraction=0.05,
    data_root=None,
    output_path=None,
    figsize=(7, 5),
):
    """Plot the extrusion-time heatmap for one simulation arrangement."""
    arrangement = arrangement.lower()
    if arrangement not in {"random", "cluster"}:
        raise ValueError("arrangement must be 'random' or 'cluster'.")
    if not 0 < remaining_fraction <= 1:
        raise ValueError("remaining_fraction must be in the interval (0, 1].")

    root = Path(data_root) if data_root is not None else Path(here())
    arrangement_config = {
        "random": {
            "directory": root / "results" / "resultados_random",
            "colour": "#709E0D",
        },
        "cluster": {
            "directory": root / "results" / "resultados_cluster",
            "colour": "#A53379",
        },
    }
    config = arrangement_config[arrangement]

    def time_to_fraction(times):
        times = np.asarray(times, dtype=float)
        n_total = len(times)
        n_needed = int(np.ceil((1 - remaining_fraction) * n_total))
        valid_times = np.sort(times[~np.isnan(times)])

        if n_total == 0 or len(valid_times) < n_needed:
            return np.nan
        return valid_times[n_needed - 1]

    data = load_data(
        str(
            config["directory"]
            / "noise_*"
            / "pct_*"
            / "sim_*"
            / "extrusion.parquet"
        ),
        columns=["mutant_id", "extrusion_time"],
    )
    if data.empty:
        raise FileNotFoundError(f"No extrusion data found for {arrangement}.")

    data = (
        data.groupby(
            ["mesh_type", "pct_mutant", "sim_id", "mutant_id"],
            as_index=False,
        )["extrusion_time"]
        .mean()
    )

    per_simulation = (
        data.groupby(["mesh_type", "pct_mutant", "sim_id"])
        ["extrusion_time"]
        .apply(time_to_fraction)
        .rename("extrusion_time")
        .reset_index()
    )

    summary = (
        per_simulation.groupby(["pct_mutant", "mesh_type"])
        ["extrusion_time"]
        .agg(mean="mean", std="std", n_successful="count")
        .reset_index()
    )
    summary["sem"] = summary["std"] / np.sqrt(summary["n_successful"])

    heatmap = summary.pivot(
        index="pct_mutant", columns="mesh_type", values="mean"
    ).sort_index(ascending=True).sort_index(axis=1, ascending=True)

    values = np.ma.masked_invalid(heatmap.to_numpy(dtype=float))
    cmap = colors.LinearSegmentedColormap.from_list(
        f"{arrangement}_heatmap",
        ["#f0f2e8", config["colour"]],
    )
    cmap.set_bad(color="white")

    fig, ax = plt.subplots(figsize=figsize)
    image = ax.imshow(
        values,
        cmap=cmap,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
    )
    ax.set_xticks(np.arange(heatmap.shape[1]))
    ax.set_yticks(np.arange(heatmap.shape[0]))
    ax.set_xticklabels(heatmap.columns)
    ax.set_yticklabels(heatmap.index)
    ax.set_xlabel("Initial mesh distortion")
    ax.set_ylabel("Mutant fraction")
    ax.set_title(arrangement.capitalize())
    fig.colorbar(image, ax=ax, label="Extrusion time")
    fig.suptitle(
        f"Time to reach {100 * remaining_fraction:g}% remaining cells",
        y=1.02,
    )
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    return summary


# t_5% (5% of mutants remaining) and t_50% (50% remaining)
random_summary = plot_extrusion_time_heatmap(
    arrangement="random", remaining_fraction=0.05,
    output_path=output_dir / "heatmap_extrusion_time_random_5pct.svg",
)
cluster_summary = plot_extrusion_time_heatmap(
    arrangement="cluster", remaining_fraction=0.05,
    output_path=output_dir / "heatmap_extrusion_time_cluster_5pct.svg",
)
random_summary = plot_extrusion_time_heatmap(
    arrangement="random", remaining_fraction=0.50,
    output_path=output_dir / "heatmap_extrusion_time_random_50pct.svg",
)
cluster_summary = plot_extrusion_time_heatmap(
    arrangement="cluster", remaining_fraction=0.50,
    output_path=output_dir / "heatmap_extrusion_time_cluster_50pct.svg",
)

## Cluster / random ratio

In [ ]:
def plot_extrusion_time_ratio_heatmap(
    remaining_fraction=0.05,
    data_root=None,
    output_path=None,
    figsize=(7, 5),
):
    """
    Plot the ratio of extrusion times:

        cluster / random

    for each combination of mutant fraction and mesh distortion.

    A ratio equal to 1 means that cluster and random have the
    same extrusion time.

    Ratio > 1:
        extrusion takes longer in cluster.

    Ratio < 1:
        extrusion takes longer in random.
    """

    if not 0 < remaining_fraction <= 1:
        raise ValueError(
            "remaining_fraction must be in the interval (0, 1]."
        )

    root = (
        Path(data_root)
        if data_root is not None
        else Path(here())
    )

    arrangement_directories = {
        "random": root / "results" / "resultados_random",
        "cluster": root / "results" / "resultados_cluster",
    }

    # ============================================================
    # Time required to reach the desired remaining fraction
    # ============================================================

    def time_to_fraction(times):

        times = np.asarray(times, dtype=float)

        n_total = len(times)

        n_needed = int(
            np.ceil((1 - remaining_fraction) * n_total)
        )

        valid_times = np.sort(
            times[~np.isnan(times)]
        )

        if n_total == 0 or len(valid_times) < n_needed:
            return np.nan

        return valid_times[n_needed - 1]

    # ============================================================
    # Load and process one arrangement
    # ============================================================

    def calculate_summary(arrangement):

        directory = arrangement_directories[arrangement]

        data = load_data(
            str(
                directory
                / "noise_*"
                / "pct_*"
                / "sim_*"
                / "extrusion.parquet"
            ),
            columns=[
                "mutant_id",
                "extrusion_time",
            ],
        )

        if data.empty:
            raise FileNotFoundError(
                f"No extrusion data found for {arrangement}."
            )

        # --------------------------------------------------------
        # Mean extrusion time per mutant
        # --------------------------------------------------------

        data = (
            data.groupby(
                [
                    "mesh_type",
                    "pct_mutant",
                    "sim_id",
                    "mutant_id",
                ],
                as_index=False,
            )["extrusion_time"]
            .mean()
        )

        # --------------------------------------------------------
        # Time needed to reach the selected remaining fraction
        # --------------------------------------------------------

        per_simulation = (
            data.groupby(
                [
                    "mesh_type",
                    "pct_mutant",
                    "sim_id",
                ]
            )["extrusion_time"]
            .apply(time_to_fraction)
            .rename("extrusion_time")
            .reset_index()
        )

        # --------------------------------------------------------
        # Average over simulations
        # --------------------------------------------------------

        summary = (
            per_simulation.groupby(
                [
                    "pct_mutant",
                    "mesh_type",
                ]
            )["extrusion_time"]
            .agg(
                mean="mean",
                std="std",
                n_successful="count",
            )
            .reset_index()
        )

        summary["sem"] = (
            summary["std"]
            / np.sqrt(summary["n_successful"])
        )

        return summary

    # ============================================================
    # Calculate summaries
    # ============================================================

    random_summary = calculate_summary("random")
    cluster_summary = calculate_summary("cluster")

    # ============================================================
    # Convert to heatmaps
    # ============================================================

    random_heatmap = (
        random_summary
        .pivot(
            index="pct_mutant",
            columns="mesh_type",
            values="mean",
        )
        .sort_index(ascending=True)
        .sort_index(axis=1, ascending=True)
    )

    cluster_heatmap = (
        cluster_summary
        .pivot(
            index="pct_mutant",
            columns="mesh_type",
            values="mean",
        )
        .sort_index(ascending=True)
        .sort_index(axis=1, ascending=True)
    )

    # ============================================================
    # Keep only combinations present in both arrangements
    # ============================================================

    common_index = random_heatmap.index.intersection(
        cluster_heatmap.index
    )

    common_columns = random_heatmap.columns.intersection(
        cluster_heatmap.columns
    )

    random_heatmap = random_heatmap.loc[
        common_index,
        common_columns,
    ]

    cluster_heatmap = cluster_heatmap.loc[
        common_index,
        common_columns,
    ]

    # ============================================================
    # Calculate cluster / random ratio
    # ============================================================

    ratio_heatmap = (
        cluster_heatmap
        / random_heatmap
    )

    # Remove infinite values
    ratio_heatmap = ratio_heatmap.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    # ============================================================
    # Prepare values
    # ============================================================

    values = np.ma.masked_invalid(
        ratio_heatmap.to_numpy(dtype=float)
    )

    if values.count() == 0:
        raise ValueError(
            "The ratio heatmap contains no valid values."
        )

    # ============================================================
    # Colour map
    # ============================================================

    cmap = sns.light_palette(
        "#7A4B41",
        as_cmap=True,
    )

    cmap.set_bad(color="white")

    # ============================================================
    # Plot
    # ============================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )

    image = ax.imshow(
        values,
        cmap=cmap,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
    )

    # ============================================================
    # Axes
    # ============================================================

    ax.set_xticks(
        np.arange(
            ratio_heatmap.shape[1]
        )
    )

    ax.set_yticks(
        np.arange(
            ratio_heatmap.shape[0]
        )
    )

    ax.set_xticklabels(
        ratio_heatmap.columns
    )

    ax.set_yticklabels(
        ratio_heatmap.index
    )

    ax.set_xlabel(
        "Initial mesh distortion"
    )

    ax.set_ylabel(
        "Mutant fraction"
    )

    ax.set_title(
        "Cluster / Random"
    )

    # ============================================================
    # Colourbar
    # ============================================================

    colorbar = fig.colorbar(
        image,
        ax=ax,
    )

    colorbar.set_label(
        "Extrusion time ratio (cluster / random)"
    )

    # ============================================================
    # Suptitle
    # ============================================================

    fig.suptitle(
        (
            f"Ratio of time to reach "
            f"{100 * remaining_fraction:g}% remaining cells"
        ),
        y=1.02,
    )

    fig.tight_layout()

    # ============================================================
    # Save
    # ============================================================

    if output_path is not None:

        fig.savefig(
            output_path,
            bbox_inches="tight",
        )

    plt.show()

    return (
        ratio_heatmap,
        random_summary,
        cluster_summary,
    )


# ================================================================
# 5% REMAINING CELLS
# ================================================================

ratio_5pct, random_5pct, cluster_5pct = (
    plot_extrusion_time_ratio_heatmap(
        remaining_fraction=0.05,
        output_path=(
            output_dir
            / "heatmap_extrusion_time_ratio_cluster_random_5pct.svg"
        ),
    )
)


# ================================================================
# 50% REMAINING CELLS
# ================================================================

ratio_50pct, random_50pct, cluster_50pct = (
    plot_extrusion_time_ratio_heatmap(
        remaining_fraction=0.50,
        output_path=(
            output_dir
            / "heatmap_extrusion_time_ratio_cluster_random_50pct.svg"
        ),
    )
)